# Polysemantic tokens v2: Perceiver bottleneck + neuron descriptors

Same 15 × 3,584 log-sum-exp pooled matrix and one token per hidden unit (its 15-value pooled-layer trajectory → Linear(15 → 128) + learned per-neuron position). Changes from v1:
* **Neuron descriptors**: each neuron's source-train mean, spread and label correlation per pooled layer (45 numbers) → small MLP → added to its token.
* **Perceiver encoder**: 64 learned 512-d latents cross-attend to the 3,584 neuron tokens (linear cost), then 2 pre-LN self-attention blocks over the latents; the mean of the latents is the constant 512-d embedding → same residual Mish 4 × 128 MLP head.
* **Optimization**: AdamW lr 3e-4 with 5% warmup + cosine decay (v1: constant 1e-3), gradient clipping 1.0, fp16.

Same six-source split, BCE, batch 128, weight decay 1e-4, seed 42, 10 epochs, best-source-validation-AUROC checkpoint, threshold calibration and held-out TriviaQA. A 10-epoch run of the original 4 × 128 log-sum-exp MLP (seed 42) is included as the matched reference.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!git checkout 177385f
!pip install -q -r requirements-kaggle.txt

In [ ]:
from pathlib import Path
SOURCE_NAME = 'sala_qwen25_g14_6src_triviaqa_v1'
domains = ['tqa', 'nq_open', 'sciq', 'popqa', 'web_questions', 'hotpotqa', 'triviaqa']
artifact_dirs = [p for p in Path('/kaggle/input').rglob(SOURCE_NAME) if p.name == SOURCE_NAME and (p / 'generation').is_dir()]
if len(artifact_dirs) != 1: raise RuntimeError(f'Expected one saved artifact set; found {artifact_dirs}')
ARTIFACT_DIR = artifact_dirs[0]
required = [*(ARTIFACT_DIR / 'generation' / d / f for d in domains for f in ('rows.jsonl', 'features.npy', 'manifest.json')), *(ARTIFACT_DIR / 'bleurt' / d / f for d in domains for f in ('scores.jsonl', 'manifest.json'))]
missing = [str(p) for p in required if not p.is_file()]
if missing: raise RuntimeError('Missing artifact files: ' + ', '.join(missing))
print('Saved features:', ARTIFACT_DIR)

In [ ]:
import subprocess, sys
RUNS = {
    'polysemantic_perceiver': [sys.executable, '/kaggle/working/SALA/sala_polysemantic.py', '--encoder', 'perceiver', '--neuron-positions', 'learned', '--latents', '64', '--latent-heads', '8', '--model-dim', '128', '--embedding-dim', '512', '--blocks', '2', '--hidden-dim', '128', '--hidden-layers', '4', '--learning-rate', '3e-4', '--lr-schedule', 'cosine'],
    'mlp_4x128_10ep': [sys.executable, '/kaggle/working/SALA/sala_raw_concat.py', '--hidden-dim', '128', '--hidden-layers', '4', '--pooling', 'logsumexp1d', '--pool-kernel-size', '3', '--pool-stride', '2'],
}
OUTPUT_DIRS = {}
for name, base in RUNS.items():
    output_dir = Path(f'/kaggle/working/sala_g14_6src_{name}')
    OUTPUT_DIRS[name] = output_dir
    command = base + ['--input-artifact-dir', str(ARTIFACT_DIR), '--output-dir', str(output_dir), '--bleurt-threshold', '0.5', '--epochs-erm', '10', '--seed', '42', '--train-device', 'cuda:0']
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
rows = []
for name, output_dir in OUTPUT_DIRS.items():
    s = json.loads((output_dir / 'summary.json').read_text())
    rows.append({'model': name, 'parameters': s.get('parameters'), 'best_epoch': s['best_epoch'],
                 'source_val_auroc': s['best_source_validation_auroc_during_training'],
                 'source_test_auroc': s['source_test_metrics']['auroc'],
                 'target_auroc': s['target_metrics']['auroc'],
                 'target_balanced_accuracy': s['target_metrics']['balanced_accuracy'],
                 'target_f1': s['target_metrics']['f1']})
display(pd.DataFrame(rows))
print(pd.DataFrame(rows).to_string())